# Batch API synthesis: temperature or feed? Sensitivity to the operating conditions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/02_sensitivity/b_operating_conditions/tutorial.ipynb)

**Global sensitivity analysis of the inputs** (`sensa` with `var_s=True`).

Which operating condition moves the product and the impurity most: the reactor temperature or the
initial concentrations? The answer tells you which inputs are worth designing and which can stay
fixed, and it is a cheap first check of a model's behaviour.

You will compute Sobol indices of the inputs over the whole operating window and in a narrow window
around a nominal recipe, and follow them along the batch.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/02_sensitivity/b_operating_conditions/' + f, f)

### The process

Batch synthesis of a quaternary ammonium salt (an intermediate of an HIV drug):

$$\mathrm{SM + TMA \rightleftharpoons QS1Cl \qquad QS1Cl \rightarrow ClDMI + MeCl}$$

The product QS1Cl forms reversibly and slowly degrades into the impurity ClDMI. Rate and
equilibrium constants follow Arrhenius / van 't Hoff laws around 296.15 K, so the model has six
parameters: three constants at the reference temperature and three energies. Measured: starting
material `y1`, product `y2`, impurity `y3`. Inputs: the temperature profile `u1` and the initial
concentrations `y10`, `y20`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from middoe.sc_sensa import sensa

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.

In [ ]:
theta_true  = [50000, 75000, 0.4116, 111900, 9905, 30000]   # the in-silico 'real' process
theta_guess = [100000, 100000, 1, 100000, 100, 10000]       # starting values
theta_min   = [10000, 0, 0.1, 50000, 10, 10000]             # bounds
theta_max   = [1000000, 200000, 10, 200000, 10000, 200000]
NAMES = ['kf_ref', 'Ef', 'kfs_ref', 'Efs', 'K_ref', 'dH']   # theta1 ... theta6 of model.py

sample = {'init': 0, 'meas': True,  # every output: measured, 17 synchronised samples at least 0.3 h apart,
          'sp': 17, 'offt': 0.3,    # always at 0 and 16 h; noise sd 0.005 mol/L
          'samp_s': 1, 'samp_f': [0, 16], 'unc': 0.005}

system = {
    'tvi': {'u1': {                       # reactor temperature [K]
        'stps': 6, 'cvp': 'LPF', 'const': 'inc',   # 6 levels joined by ramps, rising only
        'min': 296.15, 'max': 306.15,
        'offl': 0.01, 'offt': 0.3,                 # min level change [K], min time between switches [h]
    }},
    'tvo': {'y1': dict(sample), 'y2': dict(sample), 'y3': dict(sample)},  # SM, product, impurity [mol/L]
    'tii': {'y10': {'min': 0.3, 'max': 1},         # initial starting material [mol/L]
            'y20': {'min': 0.19, 'max': 1}},       # initial trimethylamine [mol/L]
    'tio': {},
    't_s': [0, 16],      # 16-h batch
    't_r': 0.1 if QUICK else 0.02,   # time grid [h] (coarser in a quick check)
    't_d': 0.3,          # dead time at both ends [h] (designs only)
}

models = {
    'can_m': ['M'],
    'krt': {'M': 'pys'},                         # a Python file with solve_model(t, y0, tii, tvi, theta)
    'src': {'M': str(Path('model.py').resolve())},
    'theta': {'M': list(theta_guess)},
    't_l': {'M': theta_min},
    't_u': {'M': theta_max},
    'mutation': {'M': [True] * 6},               # True = estimated, False = held at its value
    'theta_names': {'M': NAMES},
}

models['theta']['M'] = list(theta_true)   # the inputs are studied at the best parameters we have

### Settings

`var_s=True, par_s=False`: the parameters stay at `models['theta']`, the inputs vary. With
`var_d=False` each input is sampled over its `min`/`max` in `system`; with a float `d` between
`nominal/d` and `nominal*d` around `tii_n`, `tvi_n`.

In [ ]:
def gsa_settings(**extra):
    gsa = {'par_s': False, 'var_s': True,
           'par_d': False, 'var_d': False,        # whole operating window from system
           'samp': 2 ** 5 if QUICK else 2 ** 11,
           'multi': False if QUICK else 0.7,
           'tii_n': [0.65, 0.595], 'tvi_n': [301.15],   # nominal recipe: y10, y20 [mol/L]; T [K]
           'plt': True}
    gsa.update(extra)
    return gsa


def final_indices(res, model='M'):
    names = res['problem'][model]['names']
    cols = {}
    for y, steps in res['analysis'][model].items():
        cols[(y, 'S1')] = steps[-1]['S1']
        cols[(y, 'ST')] = steps[-1]['ST']
    return pd.DataFrame(cols, index=names).round(3)


window = sensa(gsa_settings(), models, system)
final_indices(window)

Over the whole window the initial concentrations dominate the amounts of product (more
material in, more product out), while the impurity also responds to the temperature, which speeds up
the side reaction.

### Around the nominal recipe

Plant operation rarely leaves a narrow window. With `var_d = 1.02` (±2 %) the inputs move only a
little around `tii_n` and `tvi_n`, a local view of the same question:

In [ ]:
narrow = sensa(gsa_settings(var_d=1.02), models, system)
pd.concat({'whole window': final_indices(window).xs('ST', axis=1, level=1),
           '±2 % around the recipe': final_indices(narrow).xs('ST', axis=1, level=1)}, axis=1)

A 2 % change of temperature (about 6 K) is a large change for an Arrhenius rate, a 2 % change of
a concentration is not: locally the temperature matters most.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, y, label in zip(axes, ['y1', 'y2', 'y3'], ['starting material', 'product', 'impurity']):
    steps = window['analysis']['M'][y]
    t = np.linspace(*system['t_s'], len(steps) + 1)[1:]
    ax.plot(t, np.array([s['ST'] for s in steps]))
    ax.set_title(label); ax.set_xlabel('time [h]'); ax.set_ylim(0, 1)
axes[0].set_ylabel('total index ST'); axes[0].legend(window['problem']['M']['names'], fontsize=8)
plt.tight_layout(); plt.show()

**Try:** make the temperature range wider in `system['tvi']['u1']`, or study the parameters and
the inputs together (`par_s=True, var_s=True`).